# Running GR00T N1 Inference — Hands-On Guide

## What is GR00T N1?

GR00T is a **Vision-Language-Action (VLA)** model — not an LLM. It doesn't generate text.

**Inputs:**
- **Camera images** (RGB video frames) — what the robot "sees"
- **Language instruction** — a natural language task like `"pick up the red cube"`
- **Robot state** (proprioception) — current joint positions, gripper state, etc.

**Output:**
- **Action trajectory** — a sequence of joint angle deltas (radians, m/s) that the robot should execute next

---

## GPU Requirements

| Model | VRAM Needed | Works on |
|-------|-------------|----------|
| GR00T-N1-2B | ~16GB | Free Colab T4 |
| GR00T-N1.6-3B | ~24GB | Colab Pro A100, RTX 4090 |

## Step 1: Install Dependencies

In [ ]:
# Check latest release tag and clone Isaac GR00T repository
import subprocess

# Clone the repo first
!git clone https://github.com/NVIDIA/Isaac-GR00T.git
%cd Isaac-GR00T

# Fetch tags and checkout the latest stable release
!git fetch --tags
latest_tag = subprocess.check_output(['git', 'describe', '--tags', '--abbrev=0']).decode().strip()
print(f"Latest release tag: {latest_tag}")
!git checkout {latest_tag}

# Install flash-attn from pre-built wheel (avoids compilation issues on Colab)
!pip install ninja packaging --quiet

import torch
cuda_version = torch.version.cuda.replace(".", "")[:3]  # e.g. "124"
torch_version = torch.__version__.split("+")[0]  # e.g. "2.6.0"
python_version = f"cp{subprocess.check_output(['python', '-c', 'import sys; print(f\"{sys.version_info.major}{sys.version_info.minor}\")']).decode().strip()}"
print(f"Detected: CUDA {torch.version.cuda}, PyTorch {torch_version}, Python {python_version}")

# Use pre-built wheel from flash-attn GitHub releases (much faster than compiling)
!pip install flash-attn --no-build-isolation --quiet 2>/dev/null || \
    pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu{cuda_version}torch{torch_version}cxx11abiFALSE-{python_version}-{python_version}-linux_x86_64.whl" --quiet 2>/dev/null || \
    echo "⚠️ flash-attn wheel not found for this environment, trying compilation with limited jobs..." && \
    MAX_JOBS=2 pip install flash-attn --no-build-isolation --quiet

# Now install gr00t
!pip install -e . --quiet
!pip install huggingface_hub --quiet

print(f"\n✓ Installed Isaac-GR00T @ {latest_tag}")

## Step 2: Check Your GPU

In [ ]:
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Recommendation based on VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20:
    print("\n→ Recommended: Use GR00T-N1-2B (smaller model)")
else:
    print("\n→ You can use GR00T-N1.6-3B (larger model)")

GPU: Tesla T4
VRAM: 15.6 GB

→ Recommended: Use GR00T-N1-2B (smaller model)


## Step 3: Download the Model from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

# Choose model based on your GPU:
# - Use N1-2B for T4 (16GB)
# - Use N1.6-3B for A100 (80GB) or RTX 4090 (24GB)

USE_SMALL_MODEL = True  # Set to False if you have 24GB+ VRAM

if USE_SMALL_MODEL:
    model_path = snapshot_download("nvidia/GR00T-N1-2B")
else:
    model_path = snapshot_download("nvidia/GR00T-N1.6-3B")

print(f"Model downloaded to: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.38G [00:00<?, ?B/s]

metadata.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--nvidia--GR00T-N1-2B/snapshots/fc879581ca32f4f6d6e02cf0cc80452f6b0c3873


## Step 4: Download Demo Dataset

GR00T uses the **LeRobot dataset format**. NVIDIA provides example datasets.

In [ ]:
from huggingface_hub import snapshot_download

# Download a small demo dataset with robot arm episodes
dataset_path = snapshot_download(
    "NVIDIA/GR00T-N1-Example-Data",
    repo_type="dataset"
)
print(f"Dataset at: {dataset_path}")

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69d22491-3ea47c4f340c1fe85e90e3ce;78458ea2-5335-4511-9607-a72b6c8ae533)

Repository Not Found for url: https://huggingface.co/api/datasets/NVIDIA/GR00T-N1-Example-Data/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

## Step 5: Load the GR00T Policy Model

In [ ]:
from gr00t.policy.gr00t_policy import Gr00tPolicy

# Embodiment tag tells the model which robot body it's controlling
# The demo dataset specifies which embodiment to use
policy = Gr00tPolicy(
    model_path=model_path,
    embodiment_tag="new_embodiment",  # matches the demo dataset
    backend="diffusion",
    device="cuda",
)

print("Model loaded successfully!")
print(f"Model expects action dim: {policy.policy.config.action_dim}")

## Step 6: Load a Sample Episode and Inspect the Data

In [ ]:
import numpy as np
from gr00t.data.dataset import ModalityConfig, LeRobotSingleDataset

# Load the demo dataset
modality_config = ModalityConfig.from_json(f"{dataset_path}/meta/modality.json")

dataset = LeRobotSingleDataset(
    dataset_path=dataset_path,
    modality_config=modality_config,
)

# Get one sample (contains: images, robot state, language instruction)
sample = dataset[0]

# Inspect the sample
print("=== INPUT TO THE MODEL ===")
print(f"Keys in sample: {list(sample.keys())}")
print()
for key, value in sample.items():
    if isinstance(value, np.ndarray):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, list):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {type(value).__name__}")

## Step 7: Run Inference — Get Action Predictions

In [ ]:
# The model takes the sample and predicts what the robot should do next
action = policy.get_action(sample)

print("=== OUTPUT FROM THE MODEL ===")
print(f"Action shape: {action['action'].shape}")
print(f"Action dtype: {action['action'].dtype}")
print(f"Action sample (first 5 values): {action['action'][0, :5]}")
print()
print("These are joint angle DELTAS in radians/meters.")
print("Each value tells a specific joint how much to move from its current position.")

## Step 8: Visualize Input and Predicted Actions

In [ ]:
import matplotlib.pyplot as plt

# Find the image key in the sample
image_keys = [k for k in sample.keys() if 'image' in k.lower() or 'video' in k.lower()]
print(f"Image keys found: {image_keys}")

if image_keys:
    img = sample[image_keys[0]]
    if len(img.shape) == 4:  # (T, H, W, C) - take last frame
        img = img[-1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Show the camera image (what the robot sees)
    axes[0].imshow(img)
    axes[0].set_title("Robot's View (Camera Input)")
    axes[0].axis('off')

    # Plot the predicted action trajectory
    actions = action['action']
    if len(actions.shape) == 2:
        for joint_idx in range(min(actions.shape[1], 6)):
            axes[1].plot(actions[:, joint_idx], label=f'Joint {joint_idx}')
    axes[1].set_title("Predicted Action Trajectory")
    axes[1].set_xlabel("Time Step")
    axes[1].set_ylabel("Delta (radians/meters)")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("No image keys found in sample")

## Step 9: Try Different Language Instructions

See how the model generates different action trajectories based on the task instruction.

In [ ]:
import matplotlib.pyplot as plt

instructions = [
    "pick up the object",
    "move the arm to the left",
    "place the object on the table",
]

fig, axes = plt.subplots(1, len(instructions), figsize=(6*len(instructions), 4))

for i, instruction in enumerate(instructions):
    # Modify the language instruction in the sample
    lang_keys = [k for k in sample.keys() if 'lang' in k.lower() or 'task' in k.lower() or 'instruction' in k.lower()]
    modified_sample = dict(sample)
    for key in lang_keys:
        modified_sample[key] = [instruction]

    result = policy.get_action(modified_sample)
    actions = result['action']

    if len(actions.shape) == 2:
        for joint_idx in range(min(actions.shape[1], 6)):
            axes[i].plot(actions[:, joint_idx], label=f'Joint {joint_idx}')
    axes[i].set_title(f'"{instruction}"')
    axes[i].set_xlabel("Time Step")
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

plt.tight_layout()
plt.show()

---

## Understanding the I/O (Interview Talking Points)

### Input (what the robot perceives):
```
┌─────────────────────────────────────────────┐
│  Camera Image(s)     →  "What do I see?"    │
│  (RGB, any resolution)                       │
│                                              │
│  Language Instruction →  "What should I do?" │
│  ("pick up the red cube")                    │
│                                              │
│  Proprioception      →  "Where am I now?"   │
│  (joint angles, gripper state)               │
└─────────────────────────────────────────────┘
```

### Output (what the robot should do):
```
┌─────────────────────────────────────────────┐
│  Action Trajectory   →  "How should I move?"│
│  (16 future timesteps of joint deltas)       │
│  Values in real physical units (rad, m/s)    │
│  State-relative (delta from current state)   │
└─────────────────────────────────────────────┘
```

### Architecture:
- **Vision encoder:** Cosmos-Reason-2B + Eagle3 ViT (processes the camera images)
- **Language encoder:** T5 transformer (encodes the task instruction)
- **Action head:** Diffusion Transformer (generates smooth action trajectories via iterative denoising)

### Key insight:
GR00T uses **flow matching** (a type of diffusion) for action generation — this is why the actions are smooth and physically plausible, unlike autoregressive token prediction. The model predicts **state-relative deltas** (not absolute positions), which makes it more robust to small errors accumulating over time.

---

## Troubleshooting

| Problem | Solution |
|---------|----------|
| CUDA OOM on T4 | Set `USE_SMALL_MODEL = True` to use `nvidia/GR00T-N1-2B` |
| `flash-attn` build fails | Already handled — installs pre-built wheel with auto-detected CUDA/torch/Python versions |
| `ModuleNotFoundError` | Run `!pip install -e .` again |
| Dataset download fails | Run `!huggingface-cli login` |
| Slow download | Model is ~6GB — give it a few minutes |
| No GPU available | Runtime → Change runtime type → GPU |